In [11]:
# General imports
import pandas as pd
import numpy as np
import itertools
import math
from tqdm import tqdm

# Import custom LIF SNN implementation
from LIF_SNN_network import SNNLayer

# Set random seed for reproducability
np.random.seed(42)

**Test data**

In [12]:
spiketrains = pd.read_csv("Frame_test_spiketrains.csv")

test_dist_train = [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 
                   0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
                   0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
                   0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
                   0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

test_obj_req = [
    [0, 0, 0],  # 0
    [0, 0, 0],  # 1
    [0, 0, 0],  # 2
    [0, 0, 0],  # 3
    [0, 0, 0],  # 4
    [0, 0, 0],  # 5
    [0, 0, 0],  # 6
    [0, 0, 0],  # 7
    [0, 0, 0],  # 8
    [0, 0, 0],  # 9
    [0, 0, 0],  # 10
    [0, 0, 0],  # 11
    [0, 0, 0],  # 12
    [0, 0, 0],  # 13
    [0, 0, 0],  # 14
    [1, 0, 0],  # 15 - Object detected LEFT
    [1, 0, 0],  # 16
    [1, 0, 0],  # 17
    [1, 0, 0],  # 18
    [1, 0, 0],  # 19
    [1, 0, 0],  # 20
    [1, 0, 0],  # 21
    [1, 0, 0],  # 22
    [1, 0, 0],  # 23
    [1, 0, 0],  # 24
    [0, 1, 0],  # 25 - Object detected CENTRE
    [0, 1, 0],  # 26
    [0, 1, 0],  # 27
    [0, 1, 0],  # 28
    [0, 1, 0],  # 29
    [0, 1, 0],  # 30
    [0, 1, 0],  # 31
    [0, 0, 1],  # 32 - Object detected RIGHT
    [0, 0, 1],  # 33
    [0, 0, 1],  # 34
    [0, 0, 1],  # 35
    [0, 0, 1],  # 36 
    [0, 0, 1],  # 37
    [0, 0, 1],  # 38
    [0, 0, 1],  # 39
    [0, 1, 1],  # 40 - Object detected CENTRE again
    [0, 1, 0],  # 41
    [0, 1, 0],  # 42
    [0, 1, 0],  # 43
    [0, 1, 0],  # 44
    [0, 1, 0],  # 45
    [0, 1, 0],  # 46
    [0, 1, 0],  # 47
    [0, 1, 0],  # 48
    [0, 1, 0],  # 49
]

correct_outputs = []
for i in range(len(test_obj_req)):
    obj_l, obj_c, obj_r = test_obj_req[i]
    
    if obj_r:
        correct_outputs.append(2)      # object right → turn right
    elif obj_l:
        correct_outputs.append(0)      # object left → turn left
    elif obj_c or test_dist_train[i]:
        correct_outputs.append(1)      # object center or close → forward
    else:
        correct_outputs.append(1)      # nothing → forward/explore

input_spikes = []
for i in range(len(spiketrains)):
    row = spiketrains.iloc[i].tolist() + [test_dist_train[i]] + test_obj_req[i]
    input_spikes.append(row)

# Test dataset
test_df = pd.read_csv("Test_full_inputs.csv")
test_dataset = [list(map(int, row.tolist())) for _, row in test_df.iterrows()]

test_correct_outputs = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 2, 1, 2, 1, 2, 1, 2, 1, 1, 1, 1, 1, 1]

**Parameters**

In [ ]:
# Input/Output size
n_inputs = 16
n_outputs = 3

# Training params
n_epochs = 50
n_runs = 10

# Neuron hyperparameters
decay_range = [0.75]
threshold_range = [4.0]
reset_range = [0.0]

# Synapse parameters
learning_rate_range = [0.125]
initial_weight_range = [0.3]
t_pre_range = [1, 2, 3]
t_post_range = [1, 2, 3]
tau_e_shift_range = [4]
dw_pos_range = [0.25]
dw_neg_range = [0.03125]
min_weight_range = [0.03125]
max_weight_range = [1.0]
dopamine_correct_range = [0.5, 1]
dopamine_wrong_range = [-0.5, -1]
mode_range = ['rstdp']

In [14]:
# Calculate total combinations and set up all configurations
ranges = [
    decay_range, threshold_range, reset_range, 
    learning_rate_range, initial_weight_range,
    t_pre_range, t_post_range, tau_e_shift_range,
    dw_pos_range, dw_neg_range, 
    min_weight_range, max_weight_range,
    dopamine_correct_range, dopamine_wrong_range,
    mode_range
]

# Printing the total number of configurations
total_configurations = math.prod(map(len, ranges))
print(f"Number of configurations: ", total_configurations)

Number of configurations:  36


**Logging network activity**

In [15]:
# Initialize history lists
tuning_results = []
mean_run_acc = []
epoch_acc = []
num_correct = 0

**Run hyperparameter tuning**

In [16]:
for config in tqdm(itertools.product(*ranges), total=total_configurations):
    (decay, threshold, reset, 
     learning_rate, initial_weight, 
     t_pre, t_post, tau_e_shift, 
     dw_pos, dw_neg, 
     min_weight, max_weight, 
     dopamine_correct, dopamine_wrong,
     mode) = config

    neuron_params = {"decay":decay, "threshold":threshold, "reset":reset}
    synapse_params = {"learning_rate": learning_rate, "w_init": initial_weight,
                  "t_pre": t_pre, "t_post": t_post, "tau_e_shift": tau_e_shift,
                  "dw_pos": dw_pos, "dw_neg": dw_neg,
                  "w_min": min_weight, "w_max": max_weight,
                  "mode": mode}

    all_run_accs = []
    all_runs_test_acc = []

    for r in range(n_runs):
        SNN = SNNLayer(n_inputs=n_inputs, n_outputs=n_outputs, 
                       synapse_params=synapse_params, neuron_params=neuron_params)

        epoch_acc = []
        epoch_test_acc = []

        for n in range(n_epochs):
            SNN.reset_state()
            num_correct = 0

            # --- TRAINING ---
            for current_spikes, correct_output in zip(input_spikes, correct_outputs):
                output_spikes = SNN.forward(input_spikes=current_spikes)
                winner_idx = SNN.winner_takes_all(output_spikes=output_spikes)

                if winner_idx == correct_output:
                    dopamine = dopamine_correct
                    num_correct += 1
                else:
                    dopamine = dopamine_wrong

                SNN.apply_reward(dopamine=dopamine, winner_idx=winner_idx)

            epoch_acc.append(num_correct / len(input_spikes))

            # --- TESTING ---
            SNN.reset_state()
            num_test_correct = 0

            for current_spikes, correct_output in zip(test_dataset, test_correct_outputs):
                output_spikes = SNN.forward(input_spikes=current_spikes)
                winner_idx = SNN.winner_takes_all(output_spikes=output_spikes)

                if winner_idx == correct_output:
                    num_test_correct += 1

            epoch_test_acc.append(num_test_correct / len(test_dataset))

        all_run_accs.append(np.mean(epoch_acc))
        all_runs_test_acc.append(np.mean(epoch_test_acc))

    tuning_results.append(
        neuron_params | synapse_params | {
            "dopamine_correct": dopamine_correct, 
            "dopamine_wrong": dopamine_wrong, 
            "mean_train_acc": np.mean(all_run_accs),
            "std_train_acc": np.std(all_run_accs),
            "mean_test_acc": np.mean(all_runs_test_acc),
            "std_test_acc": np.std(all_runs_test_acc),
        }
    )

100%|██████████| 36/36 [02:26<00:00,  4.07s/it]


In [17]:
df_tuning_results = pd.DataFrame(tuning_results)
df_tuning_results.to_csv("CSV_results/SNN_hyperparameter_Results.csv", index=False)
print(f"\nBest config:\n{df_tuning_results.loc[df_tuning_results['mean_test_acc'].idxmax()]}")


Best config:
decay                  0.75
threshold               4.0
reset                   0.0
learning_rate         0.125
w_init                  0.3
t_pre                     2
t_post                    3
tau_e_shift               4
dw_pos                 0.25
dw_neg              0.03125
w_min               0.03125
w_max                   1.0
mode                  rstdp
dopamine_correct        1.0
dopamine_wrong         -0.5
mean_train_acc        0.574
std_train_acc           0.0
mean_test_acc          0.66
std_test_acc            0.0
Name: 22, dtype: object


In [18]:
# Print final weights
import numpy as np
weights = SNN.get_weights()
print("Final weights (rows=output neurons, cols=inputs):")
print(np.round(weights, 3))

# Run one final pass to see per-sample decisions
print("\nPer-sample results:")
for i, (spikes, correct) in enumerate(zip(input_spikes, correct_outputs)):
    out = SNN.forward(input_spikes=spikes)
    winner = SNN.winner_takes_all(out)
    mems = [round(n.mem, 3) for n in SNN.neurons]
    print(f"  Sample {i}: winner={winner} correct={correct} {'✓' if winner==correct else '✗'} spikes={out} mems={mems}")

Final weights (rows=output neurons, cols=inputs):
[[0.031 0.136 0.146 0.031 0.031 0.063 0.076 0.08  0.1   0.191 0.369 0.271
  0.201 0.155 0.146 0.254]
 [0.698 0.461 0.44  0.724 0.594 0.473 0.574 0.529 0.319 0.643 0.196 1.
  0.189 0.193 0.941 0.625]
 [0.062 0.177 0.072 0.066 0.069 0.116 0.084 0.107 0.262 0.114 0.167 0.3
  0.18  0.23  0.233 0.271]]

Per-sample results:
  Sample 0: winner=1 correct=1 ✓ spikes=[0, 0, 0] mems=[-0.5, np.float64(1.8), np.float64(0.897)]
  Sample 1: winner=1 correct=1 ✓ spikes=[0, 0, 0] mems=[-0.5, np.float64(2.809), np.float64(0.682)]
  Sample 2: winner=1 correct=1 ✓ spikes=[0, 1, 0] mems=[-0.5, 0.0, np.float64(0.408)]
  Sample 3: winner=1 correct=1 ✓ spikes=[0, 0, 0] mems=[-0.5, np.float64(0.868), np.float64(-0.047)]
  Sample 4: winner=1 correct=1 ✓ spikes=[0, 0, 0] mems=[-0.5, np.float64(2.77), np.float64(-0.24)]
  Sample 5: winner=1 correct=1 ✓ spikes=[0, 0, 0] mems=[-0.5, np.float64(3.948), np.float64(-0.5)]
  Sample 6: winner=1 correct=1 ✓ spikes=[0, 1, 

In [19]:
# Load grid search results
df = pd.read_csv("CSV_results/SNN_hyperparameter_Results.csv")

# Top 25 configurations
top_25 = df.sort_values(by=['mean_test_acc'], ascending=[False]).head(25)

print("=== Top 25 SNN Configurations ===")
print(top_25[['mean_test_acc', 'decay', 'threshold', 'w_init', 'reset', 'learning_rate', 'mode', 
              't_pre', 't_post', 'tau_e_shift', 'dw_pos', 'dw_neg',
              'w_min', 'w_max', 'dopamine_correct', 'dopamine_wrong']])

# Parameter impact analysis
print("\n=== Impact of Decay on Accuracy ===")
print(df.groupby('decay')['mean_test_acc'].mean())

print("\n=== Impact of Threshold on Accuracy ===")
print(df.groupby('threshold')['mean_test_acc'].mean())

print("\n=== Impact of Learning Rate on Accuracy ===")
print(df.groupby('learning_rate')['mean_test_acc'].mean())

print("\n=== Impact of dw_pos on Accuracy ===")
print(df.groupby('dw_pos')['mean_test_acc'].mean())

print("\n=== Impact of dw_neg on Accuracy ===")
print(df.groupby('dw_neg')['mean_test_acc'].mean())

print("\n=== Impact of tau_e_shift on Accuracy ===")
print(df.groupby('tau_e_shift')['mean_test_acc'].mean())

print("\n=== Impact of w_init on Accuracy ===")
print(df.groupby('w_init')['mean_test_acc'].mean())

# Best overall
print("\n=== Best Overall Config ===")
best = df.sort_values('mean_test_acc', ascending=False).iloc[0]
print(best)

# Worst overall
print("\n=== Worst Overall Config ===")
best = df.sort_values('mean_test_acc', ascending=True).iloc[0]
print(best)

=== Top 25 SNN Configurations ===
    mean_test_acc  decay  threshold  w_init  reset  learning_rate   mode  \
22         0.6600   0.75        4.0     0.3    0.0          0.125  rstdp   
27         0.6570   0.75        4.0     0.3    0.0          0.125  rstdp   
35         0.6558   0.75        4.0     0.3    0.0          0.125  rstdp   
19         0.6500   0.75        4.0     0.3    0.0          0.125  rstdp   
0          0.6478   0.75        4.0     0.3    0.0          0.125  rstdp   
16         0.6476   0.75        4.0     0.3    0.0          0.125  rstdp   
30         0.6458   0.75        4.0     0.3    0.0          0.125  rstdp   
10         0.6454   0.75        4.0     0.3    0.0          0.125  rstdp   
34         0.6450   0.75        4.0     0.3    0.0          0.125  rstdp   
24         0.6446   0.75        4.0     0.3    0.0          0.125  rstdp   
14         0.6440   0.75        4.0     0.3    0.0          0.125  rstdp   
26         0.6354   0.75        4.0     0.3    0.0    